# Emerging Tech Lab - Kinetic Opentrons Protocol

## Kinetic protocol setup

In [ ]:
from kinetic_opentrons_helpers import (
    setup_run_logger,
    get_run_log_path,
    log_step,
    build_timepoint_map,
    flatten_timepoint_wells,
    validate_volume_for_p300,
    dispense_to_wells_with_tip_changes,
    set_robot_speeds,
    log_absolute_time_tick,
    log_reverse_dialdehyde_order,
    summarise_middle_replicate_finish_times,
)

run_log_path = setup_run_logger(
    log_prefix="kinetic_opentrons_run_log",
    log_dir="opentrons_logs",
)
log_step(None, f"Run log initialised: {run_log_path}")

import opentrons.execute
protocol = opentrons.execute.get_protocol_api("2.19") # Restarting the kernel again.

# Loading labware
plate_48 = protocol.load_labware(
    "greenaway_48_wellplate_3750ul",
    location = 2
)
plate_8 = protocol.load_labware(
    "greenaway_8_wellplate_20000ul",
    location = 3
)
pip_rack = protocol.load_labware(
    "opentrons_96_tiprack_300ul",
    location = 1
)

# Load Pipettes
pip_300 = protocol.load_instrument(
    "p300_single_gen2",
    "left", # Change "right" or "left" depending on which arm the 300 uL pipette is
    tip_racks = [pip_rack]
)

# -----------------------------
# User setup: liquid handling
# -----------------------------

# Optimised handling parameters for volatile organic solvent mixtures
ASPIRATE_RATE = 70          # uL/s
STANDARD_DISPENSE_RATE = 70 # uL/s
SLOW_DISPENSE_RATE = 10     # uL/s, used for dropwise dialdehyde addition
AIR_GAP_VOLUME = 15         # uL
MAX_DISPENSE = 200          # uL; two thirds of the P300 volume
PRE_WET_CYCLES = 3
PRE_WET_VOLUME = 180        # uL
TIP_CHANGE_INTERVAL = 3     # compromise: discard tip after every 3 target-well dispenses
TRANSFER_MODE = "fast"      # "fast" = one tip per reagent/solvent source; "accurate" = change tips regularly

pip_300.flow_rate.aspirate = ASPIRATE_RATE
pip_300.flow_rate.dispense = STANDARD_DISPENSE_RATE

# Speed up robot movement while keeping liquid-handling flow rates controlled.
# Rim-touching during pre-wetting is slowed separately inside the helper function.
set_robot_speeds(
    protocol=protocol,
    pipette=pip_300,
    pipette_default_speed=400,
)

# -----------------------------
# User setup: source locations
# -----------------------------

# 8-well source plate layout, can be changed if needed:
# A1: diamine stock for this kinetic run
# B1: dialdehyde stock for this kinetic run

REACTION_CONDITION = {
    "name": "chloroform",
    "diamine_source": plate_8["A1"],
    "dialdehyde_source": plate_8["B1"],
}

if REACTION_CONDITION["diamine_source"] is None or REACTION_CONDITION["dialdehyde_source"] is None:
    raise ValueError("REACTION_CONDITION source wells must not be None.")

# Kinetic target layout:
# Each nominal time point is run in triplicate.
# Replicates are arranged vertically within one column block.
#
# Time points 1-8 use the upper half of the 48-well plate:
#   time point 1 -> A1, B1, C1
#   time point 2 -> A2, B2, C2
#   ...
#   time point 8 -> A8, B8, C8
#
# Time points 9-16 use the lower half:
#   time point 9  -> D1, E1, F1
#   time point 10 -> D2, E2, F2
#   ...
#   time point 16 -> D8, E8, F8
#
# Dialdehyde is added in reverse nominal time order so that the longest
# reaction time starts first and the shortest reaction time starts last.
# Time-point labels for records only.
# The robot does not wait for these time intervals; instead, the dialdehyde addition order
# is arranged so that later sampling time points are started first and earlier time points last.
# This makes manual sampling/quenching easier: the 0 min samples are started closest to work-up.
TIMEPOINTS_MIN = [5, 15, 25, 35, 45, 55, 65, 75]

if not 1 <= len(TIMEPOINTS_MIN) <= 16:
    raise ValueError("TIMEPOINTS_MIN must contain between 1 and 16 time points to fit the vertical triplicate 48-well layout.")

# -----------------------------
# User setup: dispense volumes
# -----------------------------

# Enter the calculated volume for each stock solution per reaction vial.
# These volumes are applied to every replicate/time-point well in this run.
volume_of_diamine = 200      # uL per well
volume_of_dialdehyde = 40   # uL per well

# -----------------------------
# Build and validate the plate map
# -----------------------------

validate_volume_for_p300(volume_of_diamine, "Diamine", protocol=protocol)
validate_volume_for_p300(volume_of_dialdehyde, "Dialdehyde", protocol=protocol)

timepoint_map = build_timepoint_map(TIMEPOINTS_MIN)
target_wells = flatten_timepoint_wells(timepoint_map)

log_step(protocol, f"Reaction condition: {REACTION_CONDITION['name']}")
log_step(protocol, "Kinetic timepoint map:")
for entry in timepoint_map:
    log_step(
        protocol,
        f"  nominal {entry['time_min']} min | timepoint {entry['timepoint_index']} | "
        f"{entry['plate_region']} plate region: {', '.join(entry['wells'])}"
    )

## Basic OT-2 sanity test

In [ ]:
# -----------------------------
# Basic OT-2 sanity test
# -----------------------------

protocol.home()
pip_300.drop_tip()

log_step(protocol, "Starting basic OT-2 sanity test.")

# Test tip pickup/drop
pip_300.pick_up_tip()
log_step(protocol, "Picked up one tip successfully.")

# Test movement to source and target wells
pip_300.move_to(plate_8["A1"].top())
log_step(protocol, "Moved to source well A1 top.")

pip_300.move_to(plate_48["A1"].top())
log_step(protocol, "Moved to target well A1 top.")

pip_300.drop_tip()
log_step(protocol, "Dropped tip successfully.")

protocol.home()
log_step(protocol, "Basic OT-2 sanity test complete.")

## Automated kinetic reaction execution

In [ ]:
# -----------------------------
# Automated kinetic reaction execution
# Correct addition order:
#   1. diamine solution
#   2. dialdehyde solution, slow/dropwise
# -----------------------------

# Store dispense records for later inspection.
diamine_dispense_records = []
dialdehyde_dispense_records = []
kinetic_start_time_summary = []
reaction_name = REACTION_CONDITION["name"]

# Add diamine solution to all wells first.
# This does not start the imine reaction until the dialdehyde is later added.
diamine_dispense_records = dispense_to_wells_with_tip_changes(
    pipette=pip_300,
    protocol=protocol,
    plate=plate_48,
    source_well=REACTION_CONDITION["diamine_source"],
    target_well_names=target_wells,
    total_volume=volume_of_diamine,
    dispense_rate=STANDARD_DISPENSE_RATE,
    reagent_name=f"diamine stock ({reaction_name})",
    tip_change_interval=TIP_CHANGE_INTERVAL,
    transfer_mode=TRANSFER_MODE,
    max_dispense=MAX_DISPENSE,
    air_gap_volume=AIR_GAP_VOLUME,
    pre_wet_cycles=PRE_WET_CYCLES,
    pre_wet_volume=PRE_WET_VOLUME,
    log_each_dispense_time=False,
    log_absolute_time=False,
)

# Refined kinetic timing logic:
# The reaction starts when dialdehyde is added.
# For easier manual sampling/quenching, start the longest nominal time points first
# and the shortest/0 min time points last.
dialdehyde_ordered_wells = log_reverse_dialdehyde_order(
    timepoint_map,
    protocol=protocol,
)

log_absolute_time_tick(protocol, f"Starting dialdehyde addition for {reaction_name}.")

dialdehyde_dispense_records = dispense_to_wells_with_tip_changes(
    pipette=pip_300,
    protocol=protocol,
    plate=plate_48,
    source_well=REACTION_CONDITION["dialdehyde_source"],
    target_well_names=dialdehyde_ordered_wells,
    total_volume=volume_of_dialdehyde,
    dispense_rate=SLOW_DISPENSE_RATE,
    reagent_name=f"dialdehyde stock ({reaction_name})",
    tip_change_interval=TIP_CHANGE_INTERVAL,
    transfer_mode=TRANSFER_MODE,
    max_dispense=MAX_DISPENSE,
    air_gap_volume=AIR_GAP_VOLUME,
    pre_wet_cycles=PRE_WET_CYCLES,
    pre_wet_volume=PRE_WET_VOLUME,
    log_each_dispense_time=True,
    log_absolute_time=True,
)

kinetic_start_time_summary = summarise_middle_replicate_finish_times(
    dispense_records=dialdehyde_dispense_records,
    timepoint_map=timepoint_map,
    protocol=protocol,
)

log_absolute_time_tick(protocol, f"Finished dialdehyde addition for {reaction_name}.")

# Reset dispense rate and home the robot.
pip_300.flow_rate.dispense = STANDARD_DISPENSE_RATE

log_step(protocol, "Kinetic start-time summary:")
for record in kinetic_start_time_summary:
    log_step(
        protocol,
        f"  nominal {record['time_min']} min | timepoint {record['timepoint_index']} | "
        f"representative well {record['representative_well']} | "
        f"dispense order {record['dispense_order_index']} | start timestamp {record['finish_timestamp']}"
    )

protocol.home()